# Lab 03: Multilayer Perceptrons (MLPs) & Custom Backpropagation Engine

Welcome to Laboratory 03! In this lab, we bridge the gap between single-layer linear models and deep neural networks:
1. **Activation Functions**: Analyze ReLU, Leaky ReLU, Sigmoid, and Tanh activation mechanics and their derivative profiles.
2. **Analytical Backpropagation Engine**: Manually implement the full multi-layer matrix calculus for backpropagation and verify against `torch.autograd`.
3. **Deep MLP on Fashion-MNIST**: Build, train, and diagnose a multi-layer deep network with hidden representations.


## 1. Technical Preliminaries & Setup


In [ ]:
# Import standard PyTorch modules, torchvision datasets, and visualization tools
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# Set deterministic seed for reproducibility
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Active Device:', device)


## 2. Activation Functions & Analytical Derivatives

### Mathematical Comparison of Activation Functions
Non-linear activations break linearity, allowing neural networks to act as universal function approximators:
* **Sigmoid**: $\sigma(z) = \frac{1}{1 + e^{-z}}$, Derivative: $\sigma'(z) = \sigma(z)(1 - \sigma(z))$. Suffers from vanishing gradients when $|z| \gg 0$.
* **Tanh**: $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$, Derivative: $\tanh'(z) = 1 - \tanh^2(z)$. Zero-centered.
* **ReLU (Rectified Linear Unit)**: $\text{ReLU}(z) = \max(0, z)$, Derivative: $1$ for $z > 0$, else $0$. Prevents gradient saturation for positive inputs.


In [ ]:
# Generate input coordinate range from -5 to +5
z = torch.linspace(-5, 5, 200, requires_grad=True)

# Compute activations
y_sigmoid = torch.sigmoid(z)
y_tanh = torch.tanh(z)
y_relu = torch.relu(z)
y_leaky = torch.nn.functional.leaky_relu(z, negative_slope=0.1)

# Plot activation functions
plt.figure(figsize=(10, 5))
plt.plot(z.detach().numpy(), y_sigmoid.detach().numpy(), label='Sigmoid $\\sigma(z)$', linewidth=2)
plt.plot(z.detach().numpy(), y_tanh.detach().numpy(), label='Tanh $\\tanh(z)$', linewidth=2)
plt.plot(z.detach().numpy(), y_relu.detach().numpy(), label='ReLU $\\max(0, z)$', linewidth=2)
plt.plot(z.detach().numpy(), y_leaky.detach().numpy(), label='Leaky ReLU ($\\alpha=0.1$)', linestyle='--', linewidth=2)
plt.title('Comparison of Deep Learning Activation Functions', fontsize=14)
plt.xlabel('Input Logit $z$', fontsize=12)
plt.ylabel('Activated Output $f(z)$', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.show()


## 3. Custom Analytical Backpropagation Engine vs. PyTorch Autograd

### Mathematical Derivation of 2-Layer MLP Backpropagation
Consider a 2-layer network with input $\mathbf{X} \in \mathbb{R}^{N \times D_{in}}$, hidden layer $\mathbf{H} \in \mathbb{R}^{N \times D_{hidden}}$, and output $\hat{\mathbf{Y}} \in \mathbb{R}^{N \times D_{out}}$:
1. **Hidden Linear Pre-activation**: $\mathbf{Z}_1 = \mathbf{X} \mathbf{W}_1 + \mathbf{b}_1$
2. **Hidden Non-linear Activation**: $\mathbf{A}_1 = \text{ReLU}(\mathbf{Z}_1) = \max(0, \mathbf{Z}_1)$
3. **Output Linear Projection**: $\hat{\mathbf{Y}} = \mathbf{A}_1 \mathbf{W}_2 + \mathbf{b}_2$
4. **Mean Squared Error Loss**: $\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} \|\hat{\mathbf{Y}}^{(i)} - \mathbf{Y}^{(i)}\|^2$

### Analytical Gradients via Matrix Chain Rule:
* $\frac{\partial \mathcal{L}}{\partial \hat{\mathbf{Y}}} = \frac{2}{N} (\hat{\mathbf{Y}} - \mathbf{Y})$
* $\frac{\partial \mathcal{L}}{\partial \mathbf{W}_2} = \mathbf{A}_1^T \left( \frac{\partial \mathcal{L}}{\partial \hat{\mathbf{Y}}} \right)$
* $\frac{\partial \mathcal{L}}{\partial \mathbf{b}_2} = \sum_{\text{rows}} \frac{\partial \mathcal{L}}{\partial \hat{\mathbf{Y}}}$
* $\frac{\partial \mathcal{L}}{\partial \mathbf{A}_1} = \left( \frac{\partial \mathcal{L}}{\partial \hat{\mathbf{Y}}} \right) \mathbf{W}_2^T$
* $\frac{\partial \mathcal{L}}{\partial \mathbf{Z}_1} = \frac{\partial \mathcal{L}}{\partial \mathbf{A}_1} \odot \mathbb{I}(\mathbf{Z}_1 > 0)$
* $\frac{\partial \mathcal{L}}{\partial \mathbf{W}_1} = \mathbf{X}^T \left( \frac{\partial \mathcal{L}}{\partial \mathbf{Z}_1} \right)$
* $\frac{\partial \mathcal{L}}{\partial \mathbf{b}_1} = \sum_{\text{rows}} \frac{\partial \mathcal{L}}{\partial \mathbf{Z}_1}$


In [ ]:
# Define network dimensions: Batch Size N=4, Input Dim=3, Hidden Dim=4, Output Dim=2
N, D_in, D_hidden, D_out = 4, 3, 4, 2

# Initialize deterministic input features and ground truth labels
X = torch.randn(N, D_in)
Y = torch.randn(N, D_out)

# Initialize layer 1 weights and biases with autograd enabled
W1 = torch.randn(D_in, D_hidden, requires_grad=True)
b1 = torch.randn(D_hidden, requires_grad=True)

# Initialize layer 2 weights and biases with autograd enabled
W2 = torch.randn(D_hidden, D_out, requires_grad=True)
b2 = torch.randn(D_out, requires_grad=True)

# ----------------------------------------------------
# 1. FORWARD PASS
# ----------------------------------------------------
Z1 = X @ W1 + b1                       # Pre-activation shape: (N, D_hidden)
A1 = torch.clamp(Z1, min=0)            # ReLU activation: max(0, Z1)
Y_pred = A1 @ W2 + b2                  # Output linear projection shape: (N, D_out)
loss = torch.mean((Y_pred - Y) ** 2)   # Mean squared error loss scalar

# ----------------------------------------------------
# 2. AUTOGRAD BACKWARD PASS
# ----------------------------------------------------
loss.backward()

# ----------------------------------------------------
# 3. MANUAL ANALYTICAL BACKPROPAGATION ENGINE
# ----------------------------------------------------
with torch.no_grad():
    # Gradient of MSE Loss w.r.t Y_pred: dL/dY_pred = (2 / (N * D_out)) * (Y_pred - Y)
    grad_Y_pred = (2.0 / (N * D_out)) * (Y_pred - Y)
    
    # Layer 2 Parameter Gradients
    grad_W2_manual = A1.t() @ grad_Y_pred
    grad_b2_manual = grad_Y_pred.sum(dim=0)
    
    # Backpropagate gradient to Hidden Layer Activations A1
    grad_A1 = grad_Y_pred @ W2.t()
    
    # Backpropagate through ReLU non-linearity: dL/dZ1 = grad_A1 * (1 if Z1 > 0 else 0)
    grad_Z1 = grad_A1.clone()
    grad_Z1[Z1 <= 0] = 0
    
    # Layer 1 Parameter Gradients
    grad_W1_manual = X.t() @ grad_Z1
    grad_b1_manual = grad_Z1.sum(dim=0)

# ----------------------------------------------------
# 4. VERIFICATION & ASSERTIONS
# ----------------------------------------------------
assert torch.allclose(W2.grad, grad_W2_manual, atol=1e-5), 'W2 gradient mismatch!'
assert torch.allclose(b2.grad, grad_b2_manual, atol=1e-5), 'b2 gradient mismatch!'
assert torch.allclose(W1.grad, grad_W1_manual, atol=1e-5), 'W1 gradient mismatch!'
assert torch.allclose(b1.grad, grad_b1_manual, atol=1e-5), 'b1 gradient mismatch!'
print('[Verification Passed] All analytical gradients match PyTorch Autograd to 5 decimal places!')


## 4. Deep MLP Architecture for Fashion-MNIST Classification

### Architecture Overview: `FashionMLP`
The `FashionMLP` class implements a multi-layer deep perceptron with non-linear hidden layers:
* **Input Layer**: `Flatten` converts 2D image `(1, 28, 28)` into 784-dimensional feature vector.
* **Hidden Layer 1**: `Linear(784, 256)` followed by `ReLU()` activation.
* **Hidden Layer 2**: `Linear(256, 128)` followed by `ReLU()` activation.
* **Output Classification Layer**: `Linear(128, 10)` mapping features to 10 class logits.


In [ ]:
# Define Deep Multilayer Perceptron Architecture
class FashionMLP(nn.Module):
    """Three-layer Deep Perceptron with ReLU activations for image classification."""
    def __init__(self, input_dim: int = 784, hidden1: int = 256, hidden2: int = 128, num_classes: int = 10):
        super(FashionMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Flatten(),                         # Reshape (B, 1, 28, 28) -> (B, 784)
            nn.Linear(input_dim, hidden1),        # First fully connected layer: 784 -> 256
            nn.ReLU(),                            # Non-linear activation
            nn.Linear(hidden1, hidden2),          # Second fully connected layer: 256 -> 128
            nn.ReLU(),                            # Non-linear activation
            nn.Linear(hidden2, num_classes)       # Output classification layer: 128 -> 10 logits
        )
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

# Load Fashion-MNIST dataset with normalization
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)) # Fashion-MNIST empirical mean and standard deviation
])

fashion_train = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
fashion_test = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(fashion_train, batch_size=128, shuffle=True)
test_loader = DataLoader(fashion_test, batch_size=256, shuffle=False)

# Initialize model, loss criterion, and Adam optimizer
mlp_model = FashionMLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp_model.parameters(), lr=0.001)

print('FashionMLP Architecture Summary:\n', mlp_model)


### FashionMLP Training Loop & Validation Performance
The following cell executes training across epochs and validates test accuracy.


In [ ]:
# Train the FashionMLP model
num_epochs = 5
for epoch in range(num_epochs):
    mlp_model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = mlp_model(images)
        loss = criterion(outputs, labels)
        
        # Backward & Optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
    train_acc = (correct / total) * 100.0
    print(f'Epoch [{epoch+1}/{num_epochs}] -> Loss: {running_loss/total:.4f} | Train Acc: {train_acc:.2f}%')

# Evaluate on test set
mlp_model.eval()
test_correct, test_total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = mlp_model(images)
        _, preds = torch.max(outputs, 1)
        test_correct += (preds == labels).sum().item()
        test_total += labels.size(0)

print(f'\nFinal Fashion-MNIST Test Accuracy: {(test_correct / test_total) * 100.0:.2f}%')


## 5. Summary & Takeaways
1. **Activation Functions**: Non-linear activations are essential for enabling neural networks to learn non-linear decision boundaries.
2. **Matrix Backpropagation**: Analytical gradients calculated via the matrix chain rule match PyTorch Autograd.
3. **Multi-Layer Perceptrons**: Adding hidden layers significantly boosts representational capacity over linear models.
